In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [23]:
from simulations.benchmarks.hf import hf
from simulations.molecules import MoleculeSimulator
from solvers.evc_solver import EVCSolver

In [24]:
from src.utils.molecule_utils import compute_cc
from src.utils.procrustes_utils import (
    compute_procrustes_matrices,
    compute_cc_with_procrustes
)

## Simulator

In [25]:
hf_simulator = MoleculeSimulator(
    molecule_fun=hf,
    basis="cc-pVTZ",
    coord_scale=0.1,
    verbose=0,
)

### Reference molecule

In [26]:
include_kwargs = {
    "include_integrals": True,
    "include_hartree_fock": True,
    "include_cc": False,
    "include_coordinates": False,
    "include_all": False
}

hf_reference = hf_simulator.simulate(molecule_kwargs={"bond_distance": 1.75, "perturb": False}, **include_kwargs)

In [27]:
hf_reference["positions"]

array([[0.  , 0.  , 0.  ],
       [1.75, 0.  , 0.  ]], dtype=float32)

In [28]:
reference_determinant = hf_reference["determinant"]
reference_determinant.shape

(44, 44)

In [29]:
reference_overlap = hf_reference["overlaps"]
reference_overlap.shape

(44, 44)

### Sample and target molecules

In [30]:
sample_molecules = hf_simulator.sample(10, include_kwargs={"include_all": False})

Generating samples: 100%|██████████| 10/10 [00:00<00:00, 483.64it/s]


In [31]:
sample_procrustes_matrices = compute_procrustes_matrices(
    batched_atoms=sample_molecules["atoms"],
    batched_positions=sample_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing procrustes: 100%|██████████| 10/10 [00:07<00:00,  1.27it/s]


In [32]:
target_molecules = hf_simulator.sample(81, include_kwargs={"include_all": False})

Generating samples: 100%|██████████| 81/81 [00:00<00:00, 525.30it/s]


In [33]:
target_procrustes_matrices = compute_procrustes_matrices(
    batched_atoms=target_molecules["atoms"],
    batched_positions=target_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing procrustes: 100%|██████████| 81/81 [01:08<00:00,  1.19it/s]


In [34]:
# This replaces setup_sample().
procrustes_cc = compute_cc_with_procrustes(
    batched_atoms=sample_molecules["atoms"],
    batched_positions=sample_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing CCSD: 100%|██████████| 10/10 [00:28<00:00,  2.90s/it]


In [35]:
procrustes_cc.keys()

dict_keys(['t1', 't2', 'energies'])

In [36]:
for k, v in procrustes_cc.items():
    print(f"{k}: {v.shape}")

t1: (10, 5, 39)
t2: (10, 5, 5, 39, 39)
energies: (10,)


In [38]:
evc_solver = EVCSolver(
    t1s=procrustes_cc["t1"],
    t2s=procrustes_cc["t2"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap
)

TypeError: EVCSolver.__init__() missing 3 required positional arguments: 'all_x', 'molecule_func', and 'basis'